# Silver — limpeza, padronização e conformação

Issue #38. Lê o Bronze (Delta) e aplica qualidade de dados:

- *trim* em todos os textos e UF (estado) em maiúsculas;
- remoção de duplicatas por chave primária;
- **integridade referencial**: descarta órfãos (itens/pagamentos/reviews/
  remessas sem pedido, pedidos/endereços sem cliente, itens sem produto/
  vendedor) e zera `category_id` de produto quando a categoria não existe.

As regras espelham `sql/02_validate_source_data.sql`. Saída em Delta
conformado: `s3a://<bucket>/silver/olist/<tabela>`.

In [ ]:
import os

minio_endpoint = os.environ.get("MINIO_ENDPOINT", "http://minio:9000")
minio_access_key = os.environ.get("MINIO_ROOT_USER", "minioadmin")
minio_secret_key = os.environ.get("MINIO_ROOT_PASSWORD", "minioadmin")
bucket = os.environ.get("DATALAKE_BUCKET", "datalake")

In [ ]:
APP_NAME = "engdb-silver-olist"
EXTRA_PACKAGES = ["org.apache.hadoop:hadoop-aws:3.3.4"]

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder.appName(APP_NAME)
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.hadoop.fs.s3a.endpoint", minio_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", minio_access_key)
    .config("spark.hadoop.fs.s3a.secret.key", minio_secret_key)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
)
spark = configure_spark_with_delta_pip(builder, extra_packages=EXTRA_PACKAGES).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

In [ ]:
from pyspark.sql import functions as F

def read_bronze(t):
    return spark.read.format("delta").load(f"s3a://{bucket}/bronze/olist/{t}")

def clean_strings(df, upper_cols=()):
    for c, typ in df.dtypes:
        if typ == "string":
            df = df.withColumn(c, F.trim(F.col(c)))
            if c in upper_cols:
                df = df.withColumn(c, F.upper(F.col(c)))
    return df

def finalize(df):
    return (df.drop("_ingestion_date", "_bronze_loaded_at")
              .withColumn("_silver_loaded_at", F.current_timestamp()))

# --- entidades base (sem dependência) ---
categories = finalize(clean_strings(read_bronze("categories")).dropDuplicates(["category_id"]))
customers = finalize(clean_strings(read_bronze("customers"), ["customer_state"]).dropDuplicates(["customer_id"]))
sellers = finalize(clean_strings(read_bronze("sellers"), ["seller_state"]).dropDuplicates(["seller_id"]))

# --- products: categoria inexistente vira NULL ---
prod = clean_strings(read_bronze("products")).dropDuplicates(["product_id"])
prod = (prod.join(categories.select(F.col("category_id").alias("_vc")),
                  prod.category_id == F.col("_vc"), "left")
            .withColumn("category_id", F.when(F.col("_vc").isNotNull(), F.col("category_id")))
            .drop("_vc"))
products = finalize(prod)

# --- orders: cliente precisa existir ---
cust_ids = customers.select("customer_id")
orders = finalize(clean_strings(read_bronze("orders")).dropDuplicates(["order_id"])
                  .join(cust_ids, "customer_id", "left_semi"))
ord_ids = orders.select("order_id")

# --- addresses: cliente precisa existir ---
addresses = finalize(clean_strings(read_bronze("addresses"), ["state"]).dropDuplicates(["address_id"])
                     .join(cust_ids, "customer_id", "left_semi"))

# --- payments / reviews / shipments: pedido precisa existir ---
payments = finalize(clean_strings(read_bronze("payments")).dropDuplicates(["payment_id"])
                    .join(ord_ids, "order_id", "left_semi"))
reviews = finalize(clean_strings(read_bronze("reviews")).dropDuplicates(["review_id"])
                   .join(ord_ids, "order_id", "left_semi"))
shipments = finalize(clean_strings(read_bronze("shipments")).dropDuplicates(["shipment_id"])
                     .join(ord_ids, "order_id", "left_semi"))

# --- order_items: pedido, produto e vendedor precisam existir ---
order_items = finalize(
    clean_strings(read_bronze("order_items")).dropDuplicates(["order_item_id"])
    .join(ord_ids, "order_id", "left_semi")
    .join(products.select("product_id"), "product_id", "left_semi")
    .join(sellers.select("seller_id"), "seller_id", "left_semi"))

conformed = {
    "categories": categories, "customers": customers, "sellers": sellers,
    "products": products, "orders": orders, "addresses": addresses,
    "payments": payments, "reviews": reviews, "shipments": shipments,
    "order_items": order_items,
}
for t, df in conformed.items():
    path = f"s3a://{bucket}/silver/olist/{t}"
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path)
    print(f"[silver] {t:13} linhas={df.count():>7}")

print("Silver concluída.")
spark.stop()